In [2]:
import duckdb

In [3]:
con = duckdb.connect(database='dados_duckdb.db', read_only=False)

In [7]:
df = con.execute("SELECT * FROM bronze_z0019").fetchdf()
df.head(10)

,NATBR,MAKTX,WERKS,MAINS,LABST,nome_arquivo,data_ingestao
0,10001,PARAFUSO,BT10,100,100,z0019_1.csv,2026-03-29 19:17:13.622547
1,10002,MARTELO,BT50,100,1500,z0019_1.csv,2026-03-29 19:17:13.622547
2,10003,PREGO,BT50,100,50,z0019_1.csv,2026-03-29 19:17:13.622547
3,10004,SERRA,BT50,100,200,z0019_2.csv,2026-03-29 19:23:07.071167
4,10005,MACHADO,BT50,100,100,z0019_2.csv,2026-03-29 19:23:07.071167
5,10003,PREGO,BT50,100,60,z0019_2.csv,2026-03-29 19:23:07.071167


In [12]:
df = con.execute("""
SELECT * FROM (
    SELECT *, ROW_NUMBER() OVER (PARTITION BY natbr ORDER BY data_ingestao DESC) AS rn
    FROM bronze_z0019 
    WHERE data_ingestao >= '2026-03-29')
WHERE rn = 1
""").fetchdf()
df.head(10)

,NATBR,MAKTX,WERKS,MAINS,LABST,nome_arquivo,data_ingestao,rn
0,10001,PARAFUSO,BT10,100,100,z0019_1.csv,2026-03-29 19:17:13.622547,1
1,10002,MARTELO,BT50,100,1500,z0019_1.csv,2026-03-29 19:17:13.622547,1
2,10004,SERRA,BT50,100,200,z0019_2.csv,2026-03-29 19:23:07.071167,1
3,10005,MACHADO,BT50,100,100,z0019_2.csv,2026-03-29 19:23:07.071167,1
4,10003,PREGO,BT50,100,60,z0019_2.csv,2026-03-29 19:23:07.071167,1


In [18]:
df_final = df.drop(columns=['nome_arquivo', 'data_ingestao', 'rn'])
df_final = df_final.rename(columns={'NATBR': 'ID'})
df_final = df_final.rename(columns={'MAKTX': 'Nome'})
df_final = df_final.rename(columns={'WERKS': 'Categoria'})
df_final = df_final.rename(columns={'MAINS': 'Fornecedor'})
df_final = df_final.rename(columns={'LABST': 'Preço'})

df_final.head(10)

,ID,Nome,Categoria,Fornecedor,Preço
0,10001,PARAFUSO,BT10,100,100
1,10002,MARTELO,BT50,100,1500
2,10004,SERRA,BT50,100,200
3,10005,MACHADO,BT50,100,100
4,10003,PREGO,BT50,100,60


In [19]:
df_final.dtypes

ID            str
Nome          str
Categoria     str
Fornecedor    str
Preço         str
dtype: object

In [20]:
df2 = df_final
df2 = df2.astype(
    {
        'ID': 'int32',
        'Nome': 'string',
        'Categoria': 'string',
        'Fornecedor': 'int32',
        'Preço': 'float32'
    }
)
df2.dtypes

ID              int32
Nome           string
Categoria      string
Fornecedor      int32
Preço         float32
dtype: object

In [23]:
con.execute("""
CREATE TABLE IF NOT EXISTS produtos (
        id BIGINT,
        nome VARCHAR,
        categoria VARCHAR,      
        fornecedor BIGINT,
        preco FLOAT
    )
""")

In [24]:
con.execute("INSERT INTO produtos SELECT * FROM df2")


In [25]:
df_resultado = con.execute("SELECT * FROM produtos").fetchdf()
df_resultado.head(10)

,id,nome,categoria,fornecedor,preco
0,10001,PARAFUSO,BT10,100,100.0
1,10002,MARTELO,BT50,100,1500.0
2,10004,SERRA,BT50,100,200.0
3,10005,MACHADO,BT50,100,100.0
4,10003,PREGO,BT50,100,60.0


In [26]:
con.close()